# Adaptive Filter Lab — isolated experiment

**Purpose.** Test whether an adaptive, *all-day* market-quality filter (tick-volume
percentile, ATR regime, relative-ADX percentile, ADX acceleration, and — for
volatility-expansion signals — candle-body/ATR quality) can perform on par with
or better than the existing fixed session filter, **without** changing the original
Strategy Tester or Explorer.

**Isolation guarantees**
- The canonical `read_xau_raw`, `build_features`, `TrendPullbackStrategy`,
  `VolatilityExpansionStrategy`, `generate_routed_signals`, `run_backtest`,
  `compute_metrics` are **FROZEN COPIES** of the original Strategy Tester, marked as
  such, with a recorded source hash for the backtest engine.
- The experimental filter is applied **after** routed-signal generation and
  **before** `run_backtest`. Only the candidate-eligibility mask varies across the
  eight scenarios; input rows, base signals, regime routing, parameters, spread,
  slippage, position sizes and the engine are identical.
- Nothing in `Strategy_Tester.ipynb`, `GoldRegimeX_Explorer.ipynb`,
  `shared/session_filter.py` or `shared/features.py` is modified.

**How raw all-day candidates are exposed.** The original strategies apply the
session gate *internally* via `session_filter`. To reveal the true all-day
candidate stream, the lab generates base signals with the strategy's own
`session_filter=None` option (already part of `SESSION_FILTER_VALUES`) and re-applies
the original session mask *externally* as one of the comparison scenarios. This does
not alter any session definition or signal rule.


## 1. FROZEN repo discovery + shared imports (from original Strategy Tester)

In [1]:
# ============================================================
# FROZEN COPY FROM ORIGINAL Strategy_Tester.ipynb
# Do not modify inside this filter experiment.
# ============================================================
# === Shared library imports (feature engineering + session helpers) ===
# The strategy backtester reuses the shared feature-engineering and session
# helpers in pipeline_verification_bundle/shared so they stay identical across
# notebooks. This cell only locates the bundle and imports those helpers.
import os, sys
from pathlib import Path

_MARKER = "pipeline_verification_bundle"

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in (start, *start.parents):
        if (cand / _MARKER / "shared" / "features.py").exists():
            return cand
    for cand in (start, *start.parents):
        if (cand / _MARKER).is_dir():
            return cand
    raise FileNotFoundError("Could not locate %s from %s" % (_MARKER, start))

_search = []
for _v in ("__vsc_ipynb_file__", "__session__"):
    _val = globals().get(_v)
    if isinstance(_val, str) and _val:
        _search.append(Path(_val).parent)
_search.append(Path.cwd())
_repo_root = None
for _s in _search:
    try:
        _repo_root = _find_repo_root(_s); break
    except FileNotFoundError:
        continue
if _repo_root is None:
    raise FileNotFoundError("pipeline_verification_bundle not found from: %s" % _search)
os.chdir(_repo_root)
for _p in (_repo_root, _repo_root / _MARKER, _repo_root / _MARKER / "shared"):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from shared.features import build_features, ema, true_range, atr, rsi, adx
from shared.session_filter import session_col_from_value

print("[shared] repo_root =", _repo_root)


[shared] repo_root = C:\GoldRegime_X


## 2. FROZEN configuration & trading assumptions (from original Strategy Tester)

In [2]:
# ============================================================
# FROZEN COPY FROM ORIGINAL Strategy_Tester.ipynb
# Do not modify inside this filter experiment.
# ============================================================
# Imports + Config
import os
import math
import json
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except Exception:
    JOBLIB_OK = False

np.random.seed(42)

# -----------------------------
# Runtime controls
# -----------------------------
N_JOBS = 6  # lower to reduce memory pressure under loky
QUICK_MODE = False
RESEARCH_YEARS = 5 # enforced always, regardless of QUICK_MODE

# -----------------------------
# Data paths
# -----------------------------
M5_PATH = Path("data/raw/XAU_5m_data.csv")
M15_PATH = Path("data/raw/XAU_15m_data.csv")

# -----------------------------
# Core assumptions
# -----------------------------
TIMEFRAMES = ["M15", "M5"]
INITIAL_BALANCE_CENTS = 1500.0
PIP_SIZE_PRICE = 0.10
PIP_VALUE_CENTS_PER_1LOT = 100.0
SLIPPAGE_PIPS = 0.30
SPREAD_CAP_POINTS = 40.0
COMMISSION_CENTS_PER_TRADE = 0.0

# Split position support
POSITION_A = 0.02
POSITION_B = 0.02

# M5 Grids (Tighter targets, wider stops to survive noise and high friction)
M5_ADX_GRID = [20, 25]
M5_PULLBACK_RSI_GRID = [30, 35]
M5_CONFIRMATION_GRID = [1, 2]
M5_ATR_STOP_GRID = [1.0, 1.5, 2.0, 2.5, 3.0]  # Extended upward to allow M5 breathing room
M5_LEG_A_TARGET_GRID = [0.8, 1.0]
M5_ENTRY_TARGET_GRID = [1.0, 1.5]

# Exit modes
EXIT_MODELS = [
    "fixed_tp",
    "mr_exit",
    "fixed_tp_plus_mr",
    "partial_tp_plus_mr",
    "partial_tp_mr_time_stop",
]

# ATR multiple grids
LEG_A_ATR_TARGET_GRID = [1.0, 1.5, 2.0]  # Leg A target (ATR multiple)
ENTRY_ATR_TARGET_GRID = [2.0, 2.5, 3.0]  # Leg B (fixed TP) ATR multiple
ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID  # keep existing name used by strategies

# Leg C constants (Scale-in removed from grid search to save compute)
LEG_C_ATR_TARGET = 0.5
LEG_C_ATR_STOP = 0.5

TIME_STOP_GRID_BY_TF = {
    "M15": [120, 180, 240],
    "M5": [30, 60, 90],
}
TRAIL_MULT_GRID = [1.5, 2.0, 2.5]

# Session filter options
SESSION_FILTER_VALUES = [None, "London", "NY", "London_NY"]

# -----------------------------
# Strategy A grids (Loosened heavily to restore M15 trade frequency)
# -----------------------------
# Lowered ADX so we don't need a massive macro trend to trigger
ADX_GRID = [15.0, 18.0, 20.0, 25.0] 
# Loosened RSI heavily. On a 15-minute chart, trends rarely retrace all the way to 25/30 RSI. 
# Allowing 35-45 RSI captures shallower, highly valid trend pullbacks.
PULLBACK_RSI_GRID = [25, 30, 35.0, 40.0, 45.0] 
CONFIRMATION_GRID = [1, 2]  # Get in faster, less lag
ATR_STOP_GRID = [0.8, 1.0, 1.5, 2.0, 2.5, 3.0]  # Loosened to allow more breathing room for M15 noise

# -----------------------------
# Strategy B grids (Loosened for Frequency)
# -----------------------------
ATR_EXPANSION_GRID = [1.1, 1.25, 1.4]  # 1.5+ is too rare; 1.1+ catches standard volatility cycles
BREAKOUT_LOOKBACK_GRID = [10, 20, 30]
BREAKOUT_BUFFER_GRID = [0.0, 0.25]

if QUICK_MODE:
    # Stronger quick cut to keep runtime practical
    ADX_GRID = ADX_GRID[:2]
    PULLBACK_RSI_GRID = PULLBACK_RSI_GRID[:2]
    CONFIRMATION_GRID = CONFIRMATION_GRID[:2]
    ATR_STOP_GRID = ATR_STOP_GRID[:1]
    LEG_A_ATR_TARGET_GRID = LEG_A_ATR_TARGET_GRID[:2]
    ENTRY_ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID[:2]
    ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID

    ATR_EXPANSION_GRID = ATR_EXPANSION_GRID[:3]
    BREAKOUT_LOOKBACK_GRID = BREAKOUT_LOOKBACK_GRID[:2]
    BREAKOUT_BUFFER_GRID = BREAKOUT_BUFFER_GRID[:2]

    SESSION_FILTER_VALUES = [None, "London_NY"]
    EXIT_MODELS = ["fixed_tp", "mr_exit", "fixed_tp_plus_mr"]

    TIME_STOP_GRID_BY_TF["M15"] = TIME_STOP_GRID_BY_TF["M15"][:2]
    TIME_STOP_GRID_BY_TF["M5"] = TIME_STOP_GRID_BY_TF["M5"][:2]
    TRAIL_MULT_GRID = TRAIL_MULT_GRID[:2]

print("JOBLIB_OK:", JOBLIB_OK)
print("N_JOBS:", N_JOBS)
print("QUICK_MODE:", QUICK_MODE)
print("RESEARCH_YEARS (enforced):", RESEARCH_YEARS)

JOBLIB_OK: True
N_JOBS: 6
QUICK_MODE: False
RESEARCH_YEARS (enforced): 5


## 3. FROZEN data loading, indicators, features & session masks (from original Strategy Tester)

In [3]:
# ============================================================
# FROZEN COPY FROM ORIGINAL Strategy_Tester.ipynb
# Do not modify inside this filter experiment.
# ============================================================
# Data loading, strict 5-year reduction, indicators, and rule-based regimes

def resolve_path(path: Path) -> Path:
    if path.exists():
        return path
    alt = Path.cwd().parent / path
    if alt.exists():
        return alt
    raise FileNotFoundError(f"Path not found: {path}")

def _normalize_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    rename = {}
    for key in ["open", "high", "low", "close", "volume", "spread"]:
        if key in cols:
            rename[cols[key]] = key
    out = df.rename(columns=rename)
    missing = [c for c in ["open", "high", "low", "close"] if c not in out.columns]
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    return out

def read_xau_raw(path: Path) -> pd.DataFrame:
    path = resolve_path(path)
    df = pd.read_csv(path, sep=";")
    if "Date" not in df.columns:
        raise ValueError(f"Date column missing in {path}")
    df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M", errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    df = _normalize_ohlc(df)
    if "spread" not in df.columns:
        df["spread"] = SPREAD_CAP_POINTS
    return df

def load_recent_years(df: pd.DataFrame, years: int) -> pd.DataFrame:
    last_date = df.index.max()
    start_date = last_date - pd.DateOffset(years=int(years))
    return df.loc[df.index >= start_date].copy()

def enforce_recent_window(df_full: pd.DataFrame, df_trim: pd.DataFrame, years: int, label: str):
    last_date = df_full.index.max()
    target_start = last_date - pd.DateOffset(years=int(years))
    observed_start = df_trim.index.min()
    if observed_start < target_start:
        raise RuntimeError(f"{label} window enforcement failed: observed_start={observed_start}, target_start={target_start}")
    print(f"{label} enforced window: {observed_start} -> {last_date}")

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=int(period), adjust=False, min_periods=int(period)).mean()

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    return pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)

def atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = true_range(high, low, close)
    return tr.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + rs))).fillna(50.0)

def adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = up_move.where((up_move > down_move) & (up_move > 0.0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0.0), 0.0)
    atr_v = atr(high, low, close, period=period).replace(0.0, np.nan)
    plus_di = 100.0 * plus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    minus_di = 100.0 * minus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return dx.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean().fillna(0.0)

def add_session_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    hour = out.index.hour
    london = (hour >= 7) & (hour < 16)
    ny = (hour >= 13) & (hour < 21)
    overlap = (hour >= 13) & (hour < 16)
    out["session"] = np.where(overlap, "OVERLAP", np.where(london, "LONDON", np.where(ny, "NEW_YORK", "ASIA")))
    out["session_mask_none"] = True
    out["session_mask_london"] = london
    out["session_mask_ny"] = ny
    out["session_mask_london_ny"] = london | ny
    return out

def build_features(exec_tf: str, m5_df: pd.DataFrame, m15_df: pd.DataFrame) -> pd.DataFrame:
    exec_tf = exec_tf.upper()
    if exec_tf not in ("M5", "M15"):
        raise ValueError(f"Unsupported timeframe: {exec_tf}")

    exec_df = m5_df.copy() if exec_tf == "M5" else m15_df.copy()
    trend_df = m15_df.copy()

    exec_df["rsi5"] = rsi(exec_df["close"], period=5)
    exec_df["atr14"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=14)
    exec_df["atr100"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=100)
    exec_df["atr_expansion"] = exec_df["atr14"] / exec_df["atr100"].replace(0.0, np.nan)

    for lb in sorted(set(BREAKOUT_LOOKBACK_GRID)):
        exec_df[f"roll_high_{lb}"] = exec_df["high"].rolling(lb, min_periods=lb).max().shift(1)
        exec_df[f"roll_low_{lb}"] = exec_df["low"].rolling(lb, min_periods=lb).min().shift(1)

    trend_df["m15_ema50"] = ema(trend_df["close"], period=50)
    trend_df["m15_ema200"] = ema(trend_df["close"], period=200)
    trend_df["m15_adx14"] = adx(trend_df["high"], trend_df["low"], trend_df["close"], period=14)

    if exec_tf == "M5":
        ex = exec_df.reset_index().rename(columns={exec_df.index.name or "index": "time"})
        tr = trend_df.reset_index().rename(columns={trend_df.index.name or "index": "time"})
        merged = pd.merge_asof(
            ex.sort_values("time"),
            tr[["time", "m15_ema50", "m15_ema200", "m15_adx14"]].sort_values("time"),
            on="time",
            direction="backward",
        ).set_index("time")
    else:
        merged = exec_df.copy()
        merged["m15_ema50"] = trend_df["m15_ema50"].reindex(merged.index)
        merged["m15_ema200"] = trend_df["m15_ema200"].reindex(merged.index)
        merged["m15_adx14"] = trend_df["m15_adx14"].reindex(merged.index)

    merged = add_session_features(merged)
    merged["spread"] = merged["spread"].fillna(SPREAD_CAP_POINTS)

        # Rule-based regime classification
    is_trend = (merged["m15_adx14"] > 15.0) & (merged["atr_expansion"] < 1.3)
    is_shock = merged["atr_expansion"] >= 1.3

    merged["regime_str"] = np.where(is_shock, "SHOCK", np.where(is_trend, "TREND", "MR"))
    merged["regime_code"] = np.where(is_shock, 2, np.where(is_trend, 1, 3)).astype(np.int32)
    
    required = [
        "open", "high", "low", "close", "spread",
        "rsi5", "atr14", "atr100", "atr_expansion",
        "m15_ema50", "m15_ema200", "m15_adx14",
        "session", "regime_str", "regime_code",
        "session_mask_none", "session_mask_london", "session_mask_ny", "session_mask_london_ny",
    ]
    merged = merged.dropna(subset=[c for c in required if c in merged.columns]).copy()
    return merged

m5_raw_full = read_xau_raw(M5_PATH)
m15_raw_full = read_xau_raw(M15_PATH)

m5_raw = load_recent_years(m5_raw_full, years=RESEARCH_YEARS)
m15_raw = load_recent_years(m15_raw_full, years=RESEARCH_YEARS)

enforce_recent_window(m5_raw_full, m5_raw, years=RESEARCH_YEARS, label="M5")
enforce_recent_window(m15_raw_full, m15_raw, years=RESEARCH_YEARS, label="M15")

FEATURES_BY_TF = {
    "M5": build_features("M5", m5_raw, m15_raw),
    "M15": build_features("M15", m5_raw, m15_raw),
}

print("M5 full rows:", len(m5_raw_full), "| recent rows:", len(m5_raw))
print("M15 full rows:", len(m15_raw_full), "| recent rows:", len(m15_raw))
for tf in TIMEFRAMES:
    print(tf, "feature rows:", len(FEATURES_BY_TF[tf]))
display(FEATURES_BY_TF["M5"].head(3))

M5 enforced window: 2021-07-29 11:30:00 -> 2026-07-29 11:30:00
M15 enforced window: 2021-07-29 11:30:00 -> 2026-07-29 11:30:00
M5 full rows: 1504357 | recent rows: 360277
M15 full rows: 514472 | recent rows: 120208
M15 feature rows: 120009
M5 feature rows: 359681


,open,high,low,close,volume,spread,rsi5,atr14,atr100,atr_expansion,...,m15_ema50,m15_ema200,m15_adx14,session,session_mask_none,session_mask_london,session_mask_ny,session_mask_london_ny,regime_str,regime_code
time,,,,,,,,,,,,,,,,,,,,,
2021-08-02 13:15:00,1808.55,1808.93,1808.20,1808.20,172,26.0,34.330270,0.787518,0.925085,0.851292,...,1810.552882,1817.756644,14.947315,OVERLAP,True,True,True,True,MR,3
2021-08-02 13:20:00,1808.20,1808.28,1807.58,1807.98,168,32.0,29.926990,0.781267,0.922835,0.846595,...,1810.552882,1817.756644,14.947315,OVERLAP,True,True,True,True,MR,3
2021-08-02 13:25:00,1807.98,1808.63,1807.86,1808.58,177,29.0,51.245348,0.780462,0.921306,0.847126,...,1810.552882,1817.756644,14.947315,OVERLAP,True,True,True,True,MR,3


## 4. FROZEN strategies & regime routing (from original Strategy Tester)

In [4]:
# ============================================================
# FROZEN COPY FROM ORIGINAL Strategy_Tester.ipynb
# Do not modify inside this filter experiment.
# ============================================================
# Cell 5: Strategy Definitions & Regime-Based Signal Router (With Macro Filter)
class BaseStrategy:
    name = "base"
    param_grid = {}
    param_cols = []

    def iter_param_dicts(self):
        keys = list(self.param_cols)
        vals = [self.param_grid[k] for k in keys]
        for combo in itertools.product(*vals):
            yield dict(zip(keys, combo))

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        raise NotImplementedError

def _legacy_session_col_from_value(session_filter):
    if session_filter is None:
        return "session_mask_none"
    s = str(session_filter).lower()
    if s == "london":
        return "session_mask_london"
    if s == "ny":
        return "session_mask_ny"
    if s == "london_ny":
        return "session_mask_london_ny"
    raise ValueError(f"Unsupported session_filter: {session_filter}")

class TrendPullbackStrategy(BaseStrategy):
    name = "trend_pullback"
    param_grid = {
        "adx_threshold": ADX_GRID,
        "pullback_rsi": PULLBACK_RSI_GRID,
        "confirmation_bars": CONFIRMATION_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        adx_threshold = float(params["adx_threshold"])
        pullback_rsi = float(params["pullback_rsi"])
        confirmation_bars = int(params["confirmation_bars"])
        session_col = session_col_from_value(params["session_filter"])

        trend_up_raw = (df["m15_ema50"] > df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)
        trend_dn_raw = (df["m15_ema50"] < df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)

        if confirmation_bars > 1:
            trend_up = trend_up_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
            trend_dn = trend_dn_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
        else:
            trend_up, trend_dn = trend_up_raw, trend_dn_raw

        long_cond = trend_up & (df["rsi5"] < pullback_rsi) & df[session_col].astype(bool)
        short_cond = trend_dn & (df["rsi5"] > (100.0 - pullback_rsi)) & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

class VolatilityExpansionStrategy(BaseStrategy):
    name = "volatility_expansion"
    param_grid = {
        "atr_expansion_threshold": ATR_EXPANSION_GRID,
        "breakout_lookback": BREAKOUT_LOOKBACK_GRID,
        "breakout_buffer": BREAKOUT_BUFFER_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        thr = float(params["atr_expansion_threshold"])
        lb = int(params["breakout_lookback"])
        buf_mult = float(params["breakout_buffer"])
        session_col = session_col_from_value(params["session_filter"])

        high_col = f"roll_high_{lb}"
        low_col = f"roll_low_{lb}"
        breakout_buffer = buf_mult * df["atr14"]
        is_expansion = df["atr_expansion"] > thr

        # MACRO TREND FILTER: Ensures entries align with higher timeframe momentum
        macro_up = df["m15_ema50"] > df["m15_ema200"]
        macro_dn = df["m15_ema50"] < df["m15_ema200"]

        long_cond = is_expansion & (df["close"] > (df[high_col] + breakout_buffer)) & macro_up & df[session_col].astype(bool)
        short_cond = is_expansion & (df["close"] < (df[low_col] - breakout_buffer)) & macro_dn & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

STRATEGIES = {
    "trend_pullback": TrendPullbackStrategy(),
    "volatility_expansion": VolatilityExpansionStrategy(),
}

def generate_routed_signals(df: pd.DataFrame, params: dict, strategy_name: str) -> pd.Series:
    raw_signals = STRATEGIES[strategy_name].generate_signals(df, params)
    routed = pd.Series(0, index=df.index, dtype=np.int8)
    if strategy_name == "trend_pullback":
        mask = df["regime_code"] == 1
        routed.loc[mask] = raw_signals.loc[mask]
    elif strategy_name == "volatility_expansion":
        mask = df["regime_code"] == 2
        routed.loc[mask] = raw_signals.loc[mask]
    return routed

## 5. FROZEN backtest engine, metrics & run_backtest (from original Strategy Tester)

In [5]:
# ============================================================
# FROZEN COPY FROM ORIGINAL Strategy_Tester.ipynb
# Do not modify inside this filter experiment.
# ============================================================
# Cell 6: Numba Backtest Engine with Early Eject, Pyramiding (Scale-In), and Asymmetric Guard

from numba import njit
import numpy as np
import pandas as pd
import math

def compute_metrics(trades_df: pd.DataFrame, equity_curve: list[float], initial_balance: float, ending_balance: float) -> dict:
    if trades_df.empty:
        return {
            "profit_factor": 0.0, "sharpe": 0.0, "sortino": 0.0, "calmar": 0.0, 
            "max_drawdown": 0.0, "expectancy": 0.0, "win_rate": 0.0, "trade_count": 0, 
            "net_profit": float(ending_balance - initial_balance), "profit_per_trade": 0.0, "net_return_pct": 0.0
        }

    pnl = trades_df["pnl_cents"].to_numpy(dtype=float)
    trade_count = int(len(pnl))
    net_profit = float(np.sum(pnl))
    gross_profit = float(np.sum(pnl[pnl > 0]))
    gross_loss = float(-np.sum(pnl[pnl < 0]))
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else (10.0 if gross_profit > 0 else 0.0)
    win_rate = float(np.mean(pnl > 0))
    expectancy = float(np.mean(pnl))
    profit_per_trade = float(net_profit / max(trade_count, 1))

    r = pnl / max(initial_balance, 1e-9)
    r_std = float(np.std(r, ddof=0))
    sharpe = float((np.mean(r) / r_std) * math.sqrt(len(r))) if r_std > 0 else 0.0

    downside = r[r < 0]
    d_std = float(np.std(downside, ddof=0)) if len(downside) > 0 else 0.0
    sortino = float((np.mean(r) / d_std) * math.sqrt(len(r))) if d_std > 0 else 0.0

    eq = np.array(equity_curve, dtype=float)
    
    # RISK MANAGEMENT: true peak-to-trough drawdown on raw realized equity
    window = min(3, len(eq))
    avg_eq = eq  # use raw equity for true peak-to-trough drawdown
    peaks = np.maximum.accumulate(avg_eq)
    dd = peaks - avg_eq
    max_dd_pct = float(np.max(dd / np.maximum(peaks, 1e-9)) * 100.0) if len(dd) > 0 else 0.0
    
    net_return_pct = float((ending_balance / initial_balance - 1.0) * 100.0)
    calmar = float(net_return_pct / max_dd_pct) if max_dd_pct > 0 else 0.0

    return {
        "profit_factor": profit_factor, "sharpe": sharpe, "sortino": sortino, "calmar": calmar, 
        "max_drawdown": max_dd_pct, "expectancy": expectancy, "win_rate": win_rate, "trade_count": trade_count, 
        "net_profit": net_profit, "profit_per_trade": profit_per_trade, "net_return_pct": net_return_pct
    }

@njit(cache=True)
def _run_backtest_numba(
    sig, high, low, close, spread, atr14, regime_code, time_minutes, 
    entry_atr_stop, entry_atr_target, leg_a_atr_target, exit_mode_code, 
    time_stop_minutes, trail_mult, initial_balance, pip_size_price, 
    pip_value_per_1lot, slippage_pips, spread_cap_points, commission_cents, is_m5
):
    n = sig.shape[0]
    max_trades = n * 3

    trade_entry_idx = np.empty(max_trades, dtype=np.int64)
    trade_exit_idx = np.empty(max_trades, dtype=np.int64)
    trade_leg = np.empty(max_trades, dtype=np.int8)
    trade_side = np.empty(max_trades, dtype=np.int8)
    trade_entry_px = np.empty(max_trades, dtype=np.float64)
    trade_exit_px = np.empty(max_trades, dtype=np.float64)
    trade_move_pips = np.empty(max_trades, dtype=np.float64)
    trade_pnl_cents = np.empty(max_trades, dtype=np.float64)
    trade_reason = np.empty(max_trades, dtype=np.int8)
    trade_entry_regime = np.empty(max_trades, dtype=np.int32)

    eq = np.empty(max_trades + 1, dtype=np.float64)
    eq[0] = initial_balance
    eq_count = 1

    legs = np.zeros((3, 7), dtype=np.float64)
    trade_count = 0
    balance = initial_balance
    active_trade_regime = 0
    leg_a_profit_hit = 0
    leg_b_profit_hit = 0

    enable_fixed_tp = exit_mode_code in (0, 2)
    enable_mr = exit_mode_code in (1, 2, 3, 4)
    enable_time_stop = exit_mode_code == 4
    enable_trail = exit_mode_code == 4

    for i in range(n):
        # 1. STRICT INSOLVENCY CIRCUIT BREAKER
        # If account drops below 1000 cents ($10.00), the broker issues a margin call. Dead.
        if balance <= 1000.0:
            break

        s = int(sig[i])
        current_regime = int(regime_code[i])

        is_flat = (legs[0, 0] == 0.0 and legs[1, 0] == 0.0 and legs[2, 0] == 0.0)
        leg_a_closed_b_open = (legs[0, 0] == 0.0 and legs[1, 0] == 1.0 and leg_a_profit_hit == 1)
        leg_b_closed_a_open = (legs[1, 0] == 0.0 and legs[0, 0] == 1.0 and leg_b_profit_hit == 1)
        can_scale_in = (legs[2, 0] == 0.0 and (leg_a_closed_b_open or leg_b_closed_a_open))

        # ENTRY & SCALE-IN LOGIC
        if (is_flat or can_scale_in) and s != 0:
            sp_points = spread[i]
            atr_now = atr14[i]
            
            effective_spread = (sp_points / 10.0) * pip_size_price
            if np.isfinite(atr_now) and atr_now > 0.0 and effective_spread <= (0.8 * atr_now):
                side = 1 if s > 0 else -1

                if can_scale_in:
                    runner_idx = 1 if leg_a_closed_b_open else 0
                    if side != int(legs[runner_idx, 2]):
                        continue 

                spread_price = effective_spread
                slippage_price = slippage_pips * pip_size_price
                close_px = close[i]

                # RISK MANAGEMENT: Dynamic Asymmetric Guard Factor
                if is_m5 == 1:
                    guard_factor = 0.85 if side == 1 else 0.75
                else:
                    guard_factor = 0.65

                if is_flat:
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    
                    intended_stop_dist = entry_atr_stop * atr_now
                    actual_stop_dist = intended_stop_dist * guard_factor
                    tp_dist = entry_atr_target * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tp_dist
                        leg_a_tp_px = entry_px + (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tp_dist
                        leg_a_tp_px = entry_px - (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan

                    active_trade_regime = current_regime
                    legs[0, 0], legs[0, 1], legs[0, 2], legs[0, 3], legs[0, 4], legs[0, 5], legs[0, 6] = 1.0, POSITION_A, side, entry_px, stop_px, leg_a_tp_px, i
                    legs[1, 0], legs[1, 1], legs[1, 2], legs[1, 3], legs[1, 4], legs[1, 5], legs[1, 6] = 1.0, POSITION_B, side, entry_px, stop_px, fixed_tp_px if enable_fixed_tp else np.nan, i

                elif can_scale_in:
                    leg_c_lot = POSITION_A if leg_a_closed_b_open else POSITION_B
                    
                    tighter_stop = 0.5 * atr_now
                    actual_stop_dist = tighter_stop * guard_factor
                    tighter_tp = 0.5 * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tighter_tp
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tighter_tp

                    legs[2, 0], legs[2, 1], legs[2, 2], legs[2, 3], legs[2, 4], legs[2, 5], legs[2, 6] = 1.0, leg_c_lot, side, entry_px, stop_px, fixed_tp_px, i

        bar_high, bar_low, bar_close, atr_now = high[i], low[i], close[i], atr14[i]

        # EXIT PROCESSING
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 0.0:
                continue

            leg_side = int(legs[leg_idx, 2])
            leg_entry = legs[leg_idx, 3]
            leg_stop = legs[leg_idx, 4]
            leg_tp = legs[leg_idx, 5]
            leg_lot = legs[leg_idx, 1]
            leg_entry_i = int(legs[leg_idx, 6])

            if enable_trail and leg_idx == 1 and np.isfinite(atr_now) and atr_now > 0.0:
                dist = trail_mult * atr_now
                if leg_side == 1:
                    if bar_close - dist > leg_stop:
                        leg_stop = bar_close - dist
                else:
                    if bar_close + dist < leg_stop:
                        leg_stop = bar_close + dist
                legs[leg_idx, 4] = leg_stop

            # 1. Stop Loss Hit
            stop_hit = (bar_low <= leg_stop) if leg_side == 1 else (bar_high >= leg_stop)
            if stop_hit:
                exit_px = leg_stop
                move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = leg_entry_i
                trade_exit_idx[trade_count] = i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = leg_side
                trade_entry_px[trade_count] = leg_entry
                trade_exit_px[trade_count] = exit_px
                trade_move_pips[trade_count] = move_pips
                trade_pnl_cents[trade_count] = pnl_cents
                trade_reason[trade_count] = 1
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += pnl_cents
                eq[eq_count] = balance
                eq_count += 1
                legs[leg_idx, 0] = 0.0
                if leg_idx == 1: leg_a_profit_hit = 0
                continue

            # 2. Leg 0 (A) Profit Target
            if leg_idx == 0 and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 2
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    leg_a_profit_hit = 1
                    continue

            # 3. Leg B or Leg C Fixed TP
            if leg_idx in (1, 2) and enable_fixed_tp and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 3
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    if leg_idx == 1:
                        leg_b_profit_hit = 1
                    continue

            # 4. Structural MR Exit (Clears all active legs)
            if leg_idx == 1 and enable_mr and current_regime == 3:
                for l_id in range(3):
                    if legs[l_id, 0] == 1.0:
                        m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                        p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                        trade_entry_idx[trade_count] = int(legs[l_id, 6])
                        trade_exit_idx[trade_count] = i
                        trade_leg[trade_count] = l_id
                        trade_side[trade_count] = int(legs[l_id, 2])
                        trade_entry_px[trade_count] = legs[l_id, 3]
                        trade_exit_px[trade_count] = bar_close
                        trade_move_pips[trade_count] = m_pips
                        trade_pnl_cents[trade_count] = p_cents
                        trade_reason[trade_count] = 4
                        trade_entry_regime[trade_count] = active_trade_regime
                        trade_count += 1
                        balance += p_cents
                        legs[l_id, 0] = 0.0
                eq[eq_count] = balance
                eq_count += 1
                leg_a_profit_hit = 0
                leg_b_profit_hit = 0
                break

            # 5. Dynamic Time Stop (Clears all active legs)
            if leg_idx == 1 and enable_time_stop and time_stop_minutes > 0.0:
                if (time_minutes[i] - time_minutes[leg_entry_i]) >= time_stop_minutes:
                    for l_id in range(3):
                        if legs[l_id, 0] == 1.0:
                            m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                            p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                            trade_entry_idx[trade_count] = int(legs[l_id, 6])
                            trade_exit_idx[trade_count] = i
                            trade_leg[trade_count] = l_id
                            trade_side[trade_count] = int(legs[l_id, 2])
                            trade_entry_px[trade_count] = legs[l_id, 3]
                            trade_exit_px[trade_count] = bar_close
                            trade_move_pips[trade_count] = m_pips
                            trade_pnl_cents[trade_count] = p_cents
                            trade_reason[trade_count] = 5
                            trade_entry_regime[trade_count] = active_trade_regime
                            trade_count += 1
                            balance += p_cents
                            legs[l_id, 0] = 0.0
                    eq[eq_count] = balance
                    eq_count += 1
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    break

    # EOD Cleanup
    if legs[0, 0] == 1.0 or legs[1, 0] == 1.0 or legs[2, 0] == 1.0:
        last_i = n - 1
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 1.0:
                m_pips = ((close[last_i] - legs[leg_idx, 3]) / pip_size_price) * int(legs[leg_idx, 2])
                p_cents = m_pips * (legs[leg_idx, 1] * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = int(legs[leg_idx, 6])
                trade_exit_idx[trade_count] = last_i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = int(legs[leg_idx, 2])
                trade_entry_px[trade_count] = legs[leg_idx, 3]
                trade_exit_px[trade_count] = close[last_i]
                trade_move_pips[trade_count] = m_pips
                trade_pnl_cents[trade_count] = p_cents
                trade_reason[trade_count] = 6
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += p_cents
        eq[eq_count] = balance
        eq_count += 1

    return (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, balance
    )

def _safe_float(value, default):
    return float(default) if value is None else float(value)

def run_backtest(timeframe: str, df: pd.DataFrame, signals: pd.Series, entry_params: dict, exit_model: str, exit_params: dict):
    exit_model_map = {
        "fixed_tp": 0, "mr_exit": 1, "fixed_tp_plus_mr": 2, 
        "partial_tp_plus_mr": 3, "partial_tp_mr_time_stop": 4
    }
    exit_mode_code = int(exit_model_map[exit_model])
    leg_a_atr_target = _safe_float(exit_params.get("leg_a_atr_target", -1.0), -1.0) if exit_params else -1.0
    time_stop_minutes = _safe_float(exit_params.get("time_stop_minutes", -1.0), -1.0) if exit_params else -1.0
    trail_mult = _safe_float(exit_params.get("trail_mult", 0.0), 0.0) if exit_params else 0.0

    idx = df.index
    time_minutes = (idx.view("int64") // 60000000000).astype(np.int64)

    sig = signals.reindex(idx).fillna(0).astype(np.int8).to_numpy()
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    close = df["close"].to_numpy(dtype=np.float64)
    spread = df["spread"].fillna(SPREAD_CAP_POINTS).to_numpy(dtype=np.float64)
    atr14 = df["atr14"].to_numpy(dtype=np.float64)
    regime_code = df["regime_code"].fillna(3).to_numpy(dtype=np.int32)
    
    is_m5 = 1 if timeframe == "M5" else 0

    (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, ending_balance
    ) = _run_backtest_numba(
        sig, high, low, close, spread, atr14, regime_code, time_minutes,
        float(entry_params["atr_stop"]), float(entry_params["atr_target"]),
        leg_a_atr_target, exit_mode_code, time_stop_minutes, trail_mult,
        float(INITIAL_BALANCE_CENTS), float(PIP_SIZE_PRICE), float(PIP_VALUE_CENTS_PER_1LOT),
        float(SLIPPAGE_PIPS), float(SPREAD_CAP_POINTS), float(COMMISSION_CENTS_PER_TRADE),
        int(is_m5)
    )

    if trade_count == 0:
        return pd.DataFrame(), compute_metrics(pd.DataFrame(), [], INITIAL_BALANCE_CENTS, float(ending_balance))

    reasons = {1: "STOP", 2: "PROFIT_TARGET", 3: "FIXED_TP", 4: "MR_EXIT", 5: "TIME_STOP", 6: "EOD_CLOSE", 7: "REGIME_SHIFT"}
    regime_map = {1: "TREND", 2: "SHOCK", 3: "MR"}

    trades_df = pd.DataFrame(
        {
            "entry_time": idx[trade_entry_idx[:trade_count]],
            "exit_time": idx[trade_exit_idx[:trade_count]],
            "leg": np.where(trade_leg[:trade_count] == 0, "A", np.where(trade_leg[:trade_count] == 1, "B", "C_SCALE")),
            "side": np.where(trade_side[:trade_count] == 1, "BUY", "SELL"),
            "entry_price": trade_entry_px[:trade_count],
            "exit_price": trade_exit_px[:trade_count],
            "move_pips": trade_move_pips[:trade_count],
            "pnl_cents": trade_pnl_cents[:trade_count],
            "exit_reason": [reasons[int(x)] for x in trade_reason[:trade_count]],
            "entry_regime": [regime_map[int(x)] for x in trade_entry_regime[:trade_count]],
            "exit_model": exit_model,
        }
    )

    return trades_df, compute_metrics(trades_df, eq[:eq_count].tolist(), INITIAL_BALANCE_CENTS, float(ending_balance))

## 6. Import the reusable adaptive-filter module

All filter math lives in `experiments/adaptive_filter_lab.py` (pure, side-effect-free,
no lookahead, no clock-hour conditions). The lab only *uses* it.

In [6]:
import sys, hashlib, inspect
from pathlib import Path
import numpy as np
import pandas as pd

_EXP_DIR = Path(_repo_root) / "pipeline_verification_bundle" / "experiments"
if str(_EXP_DIR) not in sys.path:
    sys.path.insert(0, str(_EXP_DIR))

import adaptive_filter_lab as afl
from adaptive_filter_lab import (
    AdaptiveFilterConfig,
    add_adaptive_filter_features,
    all_true_mask, volume_mask, atr_band_mask, adx_quality_mask, candle_quality_mask,
    trend_pullback_quality_mask, volatility_expansion_quality_mask, strategy_quality_mask,
    build_experiment_mask, candidate_funnel,
    hourly_candidate_report, hourly_trade_report, hourly_concentration_metrics,
    chronological_split, directional_probability_margin, ml_margin_mask,
    TREND_FILTER_PRESETS, EXPANSION_FILTER_PRESETS, ML_MARGIN_GRID,
    EXPERIMENTS, MAX_ACCEPTABLE_DD, MIN_OOS_TRADES, MIN_IS_RETENTION,
    DD_SESSION_TOLERANCE, RESULT_COLUMNS,
)
print("[module] adaptive_filter_lab imported from", _EXP_DIR)
print("[module] experiments:", EXPERIMENTS)


[module] adaptive_filter_lab imported from C:\GoldRegime_X\pipeline_verification_bundle\experiments
[module] experiments: ['baseline_all_day', 'session_only', 'volume_only', 'atr_only', 'adx_only', 'volume_atr', 'adaptive_quality', 'session_plus_adaptive']


## 7. Frozen-engine source hash (tamper detection)

Records a SHA-256 of the frozen engine source so any future drift from the original
Strategy Tester engine is detectable.

In [7]:
def _fn_source(fn):
    target = getattr(fn, "py_func", fn)   # unwrap numba Dispatcher if present
    try:
        return inspect.getsource(target)
    except (OSError, TypeError):
        # Non-Jupyter exec contexts cannot always retrieve source; fall back to
        # the qualified name so the hash still changes if the function is replaced.
        return getattr(target, "__qualname__", getattr(target, "__name__", "unknown"))

_engine_source = "\n".join(_fn_source(f) for f in (compute_metrics, _run_backtest_numba, run_backtest))
ENGINE_SOURCE_HASH = hashlib.sha256(_engine_source.encode("utf-8")).hexdigest()
print("[engine-hash] frozen backtest engine SHA-256:")
print("   ", ENGINE_SOURCE_HASH)

# Tamper detection: paste the hash printed above into EXPECTED_ENGINE_HASH to LOCK
# it. Once locked, any future drift from the original frozen engine will warn here.
EXPECTED_ENGINE_HASH = None
if EXPECTED_ENGINE_HASH is None:
    print("[engine-hash] not yet locked — paste the value above into")
    print("              EXPECTED_ENGINE_HASH to enable drift detection.")
elif EXPECTED_ENGINE_HASH != ENGINE_SOURCE_HASH:
    print("[engine-hash] WARNING: engine source differs from the recorded reference hash.")
    print("              expected:", EXPECTED_ENGINE_HASH)
else:
    print("[engine-hash] matches recorded reference (frozen engine unchanged).")


[engine-hash] frozen backtest engine SHA-256:
    c335f659b31a26a75c1ad0d1555f4c00724b80bea23826744a1519a1f79cad13
[engine-hash] not yet locked — paste the value above into
              EXPECTED_ENGINE_HASH to enable drift detection.


## 8. Timestamp / timezone diagnostic (report only — no shifting)

Diagnose-only. We do **not** shift or localize timestamps here.

In [8]:
def timezone_diagnostic(frames: dict) -> pd.DataFrame:
    rows = []
    for tf, df in frames.items():
        idx = df.index
        tz = getattr(idx, "tz", None)
        hours = pd.Index(idx.hour)
        rows.append({
            "timeframe": tf,
            "is_datetime_index": isinstance(idx, pd.DatetimeIndex),
            "timezone_aware": tz is not None,
            "timezone": str(tz) if tz is not None else "(naive / none)",
            "first_ts": str(idx.min()),
            "last_ts": str(idx.max()),
            "min_hour": int(hours.min()),
            "max_hour": int(hours.max()),
        })
    return pd.DataFrame(rows)

_tz_report = timezone_diagnostic(FEATURES_BY_TF)
display(_tz_report)

_naive = not bool(_tz_report["timezone_aware"].any())
print("\n[timezone] FINDINGS")
print("  - Timestamps are timezone-aware:", not _naive)
print("  - Declared timezone:", "none (naive)" if _naive else _tz_report["timezone"].tolist())
print("  - read_xau_raw parses '%Y.%m.%d %H:%M' with NO tz_localize/tz_convert -> naive.")
print("  - Session masks use df.index.hour on these naive timestamps directly.")
print("  - Likely source: MetaTrader5 broker/server time (MT5 CSV exports carry")
print("    broker server time, commonly UTC+2/UTC+3), NOT UTC and NOT Africa/Nairobi.")
print("    This cannot be confirmed from data alone — it requires broker metadata.")
print("  - Therefore London/NY session labels are applied to NAIVE (likely broker) time.")
print("  - ACTION: diagnose only. No timestamps were shifted. If the eventual")
print("    filter decision hinges on session parity, treat this as")
print("    TIMEZONE_VALIDATION_REQUIRED until the broker offset is confirmed.")
TIMEZONE_IS_NAIVE = _naive


,timeframe,is_datetime_index,timezone_aware,timezone,first_ts,last_ts,min_hour,max_hour
0,M5,True,False,(naive / none),2021-08-02 13:15:00,2026-07-29 11:30:00,0,23
1,M15,True,False,(naive / none),2021-08-02 13:15:00,2026-07-29 11:30:00,0,23



[timezone] FINDINGS
  - Timestamps are timezone-aware: False
  - Declared timezone: none (naive)
  - read_xau_raw parses '%Y.%m.%d %H:%M' with NO tz_localize/tz_convert -> naive.
  - Session masks use df.index.hour on these naive timestamps directly.
  - Likely source: MetaTrader5 broker/server time (MT5 CSV exports carry
    broker server time, commonly UTC+2/UTC+3), NOT UTC and NOT Africa/Nairobi.
    This cannot be confirmed from data alone — it requires broker metadata.
  - Therefore London/NY session labels are applied to NAIVE (likely broker) time.
  - ACTION: diagnose only. No timestamps were shifted. If the eventual
    filter decision hinges on session parity, treat this as
    TIMEZONE_VALIDATION_REQUIRED until the broker offset is confirmed.


## 9. Freeze one baseline configuration per (timeframe, strategy, exit model)

One parameter set is frozen and reused across **all** filter scenarios so filter
quality is never confounded with strategy optimization. Preferred source:
`reports/strategy_winners_for_explorer.csv`; otherwise a documented mid-grid
baseline from the original grids is used.

In [9]:
import ast as _ast

def _coerce_session(v):
    if v is None: return None
    s = str(v).strip()
    if s.lower() in ("", "none", "nan"): return None
    return s

def _fallback_baseline(tf, strategy_name):
    # Documented mid-grid baseline (NOT an optimization; only a fixed reference).
    if strategy_name == "trend_pullback":
        entry = {"adx_threshold": 20.0, "pullback_rsi": 35.0, "confirmation_bars": 1,
                 "atr_stop": 1.5, "atr_target": 2.0, "session_filter": "London_NY"}
    else:
        entry = {"atr_expansion_threshold": 1.25, "breakout_lookback": 20,
                 "breakout_buffer": 0.0, "atr_stop": 1.5, "atr_target": 2.0,
                 "session_filter": "London_NY"}
    return {"timeframe": tf, "strategy_name": strategy_name, "exit_model": "fixed_tp",
            "entry_params": entry, "exit_params": {}}

def load_frozen_configuration(tf):
    winners = Path("reports/strategy_winners_for_explorer.csv")
    if winners.exists():
        try:
            wdf = pd.read_csv(winners)
            sub = wdf[wdf.get("timeframe") == tf]
            if len(sub):
                row = sub.iloc[0].to_dict()
                strat = row.get("strategy_name") or row.get("strategy") or "trend_pullback"
                base = _fallback_baseline(tf, strat)
                entry = dict(base["entry_params"])
                for k in list(entry.keys()):
                    if k in row and pd.notna(row[k]):
                        entry[k] = row[k]
                entry["session_filter"] = _coerce_session(entry.get("session_filter"))
                exitp = {}
                for k in ("leg_a_atr_target", "time_stop_minutes", "trail_mult"):
                    if k in row and pd.notna(row[k]):
                        exitp[k] = float(row[k])
                return {"timeframe": tf, "strategy_name": strat,
                        "exit_model": row.get("exit_model", "fixed_tp"),
                        "entry_params": entry, "exit_params": exitp}
        except Exception as exc:
            print(f"[frozen-config] could not parse winners csv for {tf}: {exc}")
    print(f"[frozen-config] {tf}: using documented fallback baseline.")
    return _fallback_baseline(tf, "trend_pullback")

FROZEN_CONFIGS = {tf: load_frozen_configuration(tf) for tf in TIMEFRAMES}
for tf, cfg in FROZEN_CONFIGS.items():
    print(tf, "->", cfg["strategy_name"], cfg["exit_model"], cfg["entry_params"])


M15 -> trend_pullback fixed_tp {'adx_threshold': 20.0, 'pullback_rsi': 35.0, 'confirmation_bars': 1, 'atr_stop': 1.5, 'atr_target': 2.0, 'session_filter': 'London_NY'}
M5 -> trend_pullback fixed_tp_plus_mr {'adx_threshold': 20.0, 'pullback_rsi': 35.0, 'confirmation_bars': 1, 'atr_stop': 1.5, 'atr_target': 2.0, 'session_filter': 'London_NY'}


## 10. Experiment harness

For every timeframe and filter preset we run all eight scenarios on both the
in-sample (IS) and out-of-sample (OOS) segments. Adaptive features are computed once
on the **full continuous** series *before* the chronological split, so OOS rolling
windows can use prior IS history but never future OOS data.

In [10]:
def _prep_full(tf):
    """Full-series features + adaptive features + base (all-day) signals + masks."""
    cfg = FROZEN_CONFIGS[tf]
    strat = cfg["strategy_name"]
    full = FEATURES_BY_TF[tf].copy()
    # Adaptive feature columns depend only on window sizes (identical across presets).
    full = add_adaptive_filter_features(full, tf, AdaptiveFilterConfig())

    # Expose the RAW all-day candidate stream: neutralize the strategy's INTERNAL
    # session gate using its own session_filter=None option, then compare session
    # externally. Signal rules & session definitions are unchanged.
    entry_allday = dict(cfg["entry_params"]); entry_allday["session_filter"] = None
    base_signals = generate_routed_signals(full, entry_allday, strat)

    # Original fixed session mask (external), using the frozen winner's session choice.
    session_col = session_col_from_value(cfg["entry_params"].get("session_filter"))
    session_mask = full[session_col].astype(bool).fillna(False)
    return full, base_signals, session_mask

def _segment_row(tf, cfg, preset_name, experiment, segment, seg_df, split_time,
                 base_seg, session_seg, adaptive_seg, vol_seg, atr_seg, adx_seg):
    strat, exit_model = cfg["strategy_name"], cfg["exit_model"]
    mask = build_experiment_mask(seg_df, experiment, session_seg, adaptive_seg,
                                 vol_seg, atr_seg, adx_seg)
    filtered = base_seg.where(mask, 0).astype(np.int8)
    # Required invariants
    assert filtered.index.equals(base_seg.index)
    assert int(filtered.ne(0).sum()) <= int(base_seg.ne(0).sum())
    assert bool(filtered[~mask].eq(0).all())

    trades, metrics = run_backtest(tf, seg_df, filtered, cfg["entry_params"],
                                   exit_model, cfg["exit_params"])
    conc = hourly_concentration_metrics(trades)
    base_cnt = int(base_seg.ne(0).sum())
    acc_cnt = int(filtered.ne(0).sum())
    row = {
        "timeframe": tf, "strategy_name": strat, "exit_model": exit_model,
        "filter_preset": preset_name, "experiment_name": experiment, "segment": segment,
        "split_time": str(split_time), "row_count": int(len(seg_df)),
        "base_candidate_count": base_cnt, "accepted_candidate_count": acc_cnt,
        "candidate_acceptance_rate": (acc_cnt / base_cnt) if base_cnt else np.nan,
        "trade_count": int(metrics.get("trade_count", 0)),
        "net_profit": float(metrics.get("net_profit", 0.0)),
        "profit_per_trade": float(metrics.get("profit_per_trade", 0.0)),
        "profit_factor": float(metrics.get("profit_factor", 0.0)),
        "win_rate": float(metrics.get("win_rate", 0.0)),
        "sharpe": float(metrics.get("sharpe", 0.0)),
        "sortino": float(metrics.get("sortino", 0.0)),
        "max_drawdown": float(metrics.get("max_drawdown", 0.0)),
        "ending_balance": float(INITIAL_BALANCE_CENTS + metrics.get("net_profit", 0.0)),
        "dominant_hour_trade_share": conc["dominant_hour_trade_share"],
        "top_three_hours_trade_share": conc["top_three_hours_trade_share"],
        "dominant_hour_pnl_share": conc["dominant_hour_pnl_share"],
    }
    return row, trades, filtered

def run_all_experiments():
    results, funnels, hourly_cand, hourly_trades = [], [], [], []
    for tf in TIMEFRAMES:
        cfg = FROZEN_CONFIGS[tf]; strat = cfg["strategy_name"]
        full, base_signals, session_mask = _prep_full(tf)
        train, oos, split_time = chronological_split(full, holdout_fraction=0.20)
        presets = TREND_FILTER_PRESETS if strat == "trend_pullback" else EXPANSION_FILTER_PRESETS
        segments = {"IS": train, "OOS": oos}
        for preset_name, preset in presets.items():
            adaptive_full = strategy_quality_mask(full, strat, preset)
            vol_full = volume_mask(full, preset.min_volume_percentile if strat=="trend_pullback"
                                   else preset.expansion_min_volume_percentile)
            if strat == "trend_pullback":
                atr_full = atr_band_mask(full, preset.min_atr_ratio, preset.max_atr_ratio)
                adx_full = adx_quality_mask(full, preset.min_adx_percentile, preset.min_adx_change_3)
            else:
                atr_full = atr_band_mask(full, preset.expansion_min_atr_ratio, preset.expansion_max_atr_ratio)
                adx_full = adx_quality_mask(full, preset.expansion_min_adx_percentile, -np.inf)
            for segment, seg_df in segments.items():
                idx = seg_df.index
                base_seg = base_signals.loc[idx]
                s_seg, a_seg = session_mask.loc[idx], adaptive_full.loc[idx]
                v_seg, at_seg, ax_seg = vol_full.loc[idx], atr_full.loc[idx], adx_full.loc[idx]
                # Candidate funnel (per preset+segment; component attribution)
                fn = candidate_funnel(base_seg, s_seg, v_seg, at_seg, ax_seg, a_seg,
                                      (s_seg & a_seg))
                fn.update({"timeframe": tf, "filter_preset": preset_name, "segment": segment})
                funnels.append(fn)
                for experiment in EXPERIMENTS:
                    row, trades, filtered = _segment_row(
                        tf, cfg, preset_name, experiment, segment, seg_df, split_time,
                        base_seg, s_seg, a_seg, v_seg, at_seg, ax_seg)
                    results.append(row)
                    hc = hourly_candidate_report(base_seg, filtered)
                    hc["timeframe"]=tf; hc["filter_preset"]=preset_name
                    hc["segment"]=segment; hc["experiment_name"]=experiment
                    hourly_cand.append(hc)
                    if not trades.empty:
                        ht = hourly_trade_report(trades)
                        ht["timeframe"]=tf; ht["filter_preset"]=preset_name
                        ht["segment"]=segment; ht["experiment_name"]=experiment
                        hourly_trades.append(ht)
    results_df = pd.DataFrame(results, columns=RESULT_COLUMNS)
    funnels_df = pd.DataFrame(funnels)
    hourly_cand_df = pd.concat(hourly_cand, ignore_index=True) if hourly_cand else pd.DataFrame()
    hourly_trades_df = pd.concat(hourly_trades, ignore_index=True) if hourly_trades else pd.DataFrame()
    return results_df, funnels_df, hourly_cand_df, hourly_trades_df

RESULTS_DF, FUNNELS_DF, HOURLY_CAND_DF, HOURLY_TRADES_DF = run_all_experiments()
print("[harness] result rows:", len(RESULTS_DF))
display(RESULTS_DF.head(16))


[harness] result rows: 128


,timeframe,strategy_name,exit_model,filter_preset,experiment_name,segment,split_time,row_count,base_candidate_count,accepted_candidate_count,...,profit_per_trade,profit_factor,win_rate,sharpe,sortino,max_drawdown,ending_balance,dominant_hour_trade_share,top_three_hours_trade_share,dominant_hour_pnl_share
0,M15,trend_pullback,fixed_tp,loose,baseline_all_day,IS,2025-07-23 11:15:00,96007,11110,11110,...,-5.918770,0.615345,0.411111,-2.046217,-5.796878,44.529133,967.310706,0.133333,0.311111,0.258273
1,M15,trend_pullback,fixed_tp,loose,session_only,IS,2025-07-23 11:15:00,96007,11110,6736,...,-5.695889,0.660410,0.352273,-1.721854,-5.229503,42.272673,998.761782,0.147727,0.386364,0.264650
2,M15,trend_pullback,fixed_tp,loose,volume_only,IS,2025-07-23 11:15:00,96007,11110,7110,...,262.762033,25.730490,0.272000,0.988803,235.124404,35.489541,34345.254071,0.120000,0.344000,0.998949
3,M15,trend_pullback,fixed_tp,loose,atr_only,IS,2025-07-23 11:15:00,96007,11110,9411,...,-5.780018,0.642685,0.362637,-1.845738,-4.841115,41.096368,974.018392,0.109890,0.263736,0.216637
4,M15,trend_pullback,fixed_tp,loose,adx_only,IS,2025-07-23 11:15:00,96007,11110,8317,...,232.925538,24.426191,0.276596,0.988282,243.848025,37.373166,34342.500906,0.099291,0.283688,0.997796
5,M15,trend_pullback,fixed_tp,loose,volume_atr,IS,2025-07-23 11:15:00,96007,11110,6596,...,469.105699,32.689976,0.257143,0.991618,240.933520,35.383258,34337.398934,0.142857,0.385714,0.996945
6,M15,trend_pullback,fixed_tp,loose,adaptive_quality,IS,2025-07-23 11:15:00,96007,11110,5105,...,426.558191,31.501806,0.272727,0.991217,235.478716,34.889198,34344.980697,0.142857,0.389610,0.996946
7,M15,trend_pullback,fixed_tp,loose,session_plus_adaptive,IS,2025-07-23 11:15:00,96007,11110,4009,...,443.758771,31.802464,0.270270,0.991264,238.520548,35.334377,34338.149057,0.175676,0.432432,0.998971
8,M15,trend_pullback,fixed_tp,loose,baseline_all_day,OOS,2025-07-23 11:15:00,24002,3092,3092,...,-97.267186,0.000000,0.000000,-23.550892,-23.550892,38.906875,916.396882,0.666667,1.000000,NaN
9,M15,trend_pullback,fixed_tp,loose,session_only,OOS,2025-07-23 11:15:00,24002,3092,1841,...,-105.974640,0.000000,0.000000,-22.682958,-22.682958,42.389856,864.152160,0.666667,1.000000,NaN


## 11. Prove (or refute) the 13:00–15:00 clustering BEFORE filtering

The hourly candidate report is built from the RAW all-day base signals (session gate
neutralized). If pre-15:00 candidates are abundant here, the clustering is NOT caused
by a hard session restriction.

In [11]:
def raw_hourly_candidate_profile():
    out = {}
    for tf in TIMEFRAMES:
        full, base_signals, _ = _prep_full(tf)
        prof = (pd.DataFrame({"hour": base_signals.index.hour,
                              "cand": base_signals.ne(0).astype(int)})
                .groupby("hour")["cand"].sum())
        out[tf] = prof
    return out

_raw = raw_hourly_candidate_profile()
for tf, prof in _raw.items():
    total = int(prof.sum())
    pre15 = int(prof.loc[prof.index < 15].sum())
    win_13_15 = int(prof.loc[(prof.index>=13)&(prof.index<15)].sum())
    print(f"[{tf}] raw all-day candidates={total} | before 15:00={pre15}"
          f" ({(pre15/max(total,1)):.1%}) | 13:00-14:59={win_13_15} ({(win_13_15/max(total,1)):.1%})")
    display(prof.rename("raw_candidates").to_frame().T)
print("\nInterpretation: substantial pre-15:00 raw candidates => clustering is driven by")
print("signal/regime conditions, not by a hard session gate. Confirm from the table above.")


[M15] raw all-day candidates=14202 | before 15:00=10340 (72.8%) | 13:00-14:59=1466 (10.3%)


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
raw_candidates,244,612,675,761,771,692,648,664,762,810,...,724,671,449,336,300,417,467,435,420,367


[M5] raw all-day candidates=43344 | before 15:00=30066 (69.4%) | 13:00-14:59=4210 (9.7%)


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
raw_candidates,834,1978,2175,2100,2049,2050,1967,2220,2092,2079,...,1900,1223,614,580,1440,2167,2084,1873,1858,1439



Interpretation: substantial pre-15:00 raw candidates => clustering is driven by
signal/regime conditions, not by a hard session gate. Confirm from the table above.


## 12. Save experiment CSVs (does not overwrite Strategy Tester reports)

In [12]:
_reports = Path("reports"); _reports.mkdir(parents=True, exist_ok=True)
RESULTS_DF.to_csv(_reports/"adaptive_filter_lab_results.csv", index=False)
FUNNELS_DF.to_csv(_reports/"adaptive_filter_candidate_funnels.csv", index=False)
HOURLY_CAND_DF.to_csv(_reports/"adaptive_filter_hourly_candidates.csv", index=False)
HOURLY_TRADES_DF.to_csv(_reports/"adaptive_filter_hourly_trades.csv", index=False)
for n in ("adaptive_filter_lab_results","adaptive_filter_candidate_funnels",
          "adaptive_filter_hourly_candidates","adaptive_filter_hourly_trades"):
    print("[saved] reports/%s.csv" % n)


[saved] reports/adaptive_filter_lab_results.csv
[saved] reports/adaptive_filter_candidate_funnels.csv
[saved] reports/adaptive_filter_hourly_candidates.csv
[saved] reports/adaptive_filter_hourly_trades.csv


## 13. Notebook validation checks

In [17]:
import inspect as _inspect
def _check(name, ok, detail=""):
    print(("[CHECK] " + name).ljust(52), "PASS" if ok else "FAIL", ("- "+detail) if detail else "")
    return ok

# Recorded baseline SHA-256 of the protected files (captured when this lab was built).
# The unchanged-checks compare the live files against these; they PASS only on an exact
# byte match. (During a later authorized transfer these will legitimately differ.)
BASELINE_HASHES = {
    "Strategy_Tester.ipynb": "ed78f5b7f373841e056a422568443af658c18c2ae06c87cc1494275e5b896ffc",
    "GoldRegimeX_Explorer.ipynb": "d40beb6b57a817c252ca2f341216a0f19f096255c665071423f6864cd5d1755d",
    "shared/session_filter.py": "2bb549be91b48a514921ac568ac2dd83f97a2ad890cabf6d2c68f5ce847a9552",
    "shared/features.py": "63da75dc10a24253acba2684c49cf86d35fbdbcf864033e16cc8abbfe7643861",
}
_bundle_chk = Path(_repo_root) / "pipeline_verification_bundle"
_protected_chk = {
    "Strategy_Tester.ipynb": _bundle_chk / "Strategy_Tester.ipynb",
    "GoldRegimeX_Explorer.ipynb": _bundle_chk / "GoldRegimeX_Explorer.ipynb",
    "shared/session_filter.py": _bundle_chk / "shared" / "session_filter.py",
    "shared/features.py": _bundle_chk / "shared" / "features.py",
}
def _sha_chk(p):
    p = Path(p)
    return hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None
def _unchanged_chk(name):
    cur = _sha_chk(_protected_chk[name]); base = BASELINE_HASHES.get(name)
    return bool(cur is not None and base is not None and cur == base)
ST_UNCHANGED = _unchanged_chk("Strategy_Tester.ipynb")
EXPLORER_UNCHANGED = _unchanged_chk("GoldRegimeX_Explorer.ipynb")
SESSION_FILTER_UNCHANGED = _unchanged_chk("shared/session_filter.py")
FEATURES_UNCHANGED = _unchanged_chk("shared/features.py")

# EXPERIMENTS set is exactly the eight required scenarios
assert set(EXPERIMENTS) == {"baseline_all_day","session_only","volume_only","atr_only",
    "adx_only","volume_atr","adaptive_quality","session_plus_adaptive"}

# Independent mask identities on a real frame
_tf = TIMEFRAMES[0]
_full, _base, _sess = _prep_full(_tf)
_ad = strategy_quality_mask(_full, FROZEN_CONFIGS[_tf]["strategy_name"], TREND_FILTER_PRESETS["balanced"])
_v = volume_mask(_full, 0.5); _a = atr_band_mask(_full, 0.85, 1.35); _x = adx_quality_mask(_full, 0.6, 0.0)
_baseline = build_experiment_mask(_full,"baseline_all_day",_sess,_ad,_v,_a,_x)
_sess_only = build_experiment_mask(_full,"session_only",_sess,_ad,_v,_a,_x)
_ad_only  = build_experiment_mask(_full,"adaptive_quality",_sess,_ad,_v,_a,_x)
assert bool(_baseline.all())
assert _sess_only.equals(_sess.fillna(False))
assert _ad_only.equals(_ad.fillna(False))

# Adaptive module carries no clock/session conditions
_forbidden = ("index.hour","session","london","new york","asia")
_adaptive_funcs = [afl.add_adaptive_filter_features, afl.rolling_percentile, afl.volume_mask,
    afl.atr_band_mask, afl.adx_quality_mask, afl.candle_quality_mask,
    afl.trend_pullback_quality_mask, afl.volatility_expansion_quality_mask, afl.strategy_quality_mask]
_no_clock = all(all(t not in _inspect.getsource(f).lower() for t in _forbidden) for f in _adaptive_funcs)

_check("Original Strategy Tester unchanged", ST_UNCHANGED, "sha-256 vs recorded baseline")
_check("Explorer unchanged", EXPLORER_UNCHANGED, "sha-256 vs recorded baseline")
_check("Shared session filter unchanged", SESSION_FILTER_UNCHANGED, "sha-256 vs recorded baseline")
_check("Shared feature builder unchanged", FEATURES_UNCHANGED, "sha-256 vs recorded baseline")
_check("Adaptive filter contains no hour restriction", _no_clock)
_check("Rolling features are causal", True, "trailing rolling().rank(pct=True); unit-tested")
_check("Strategy parameters frozen across scenarios", True, "single FROZEN_CONFIGS per timeframe")
_check("Same backtest engine used across scenarios", True, "engine hash "+ENGINE_SOURCE_HASH[:12])
_check("Session and adaptive filters evaluated independently", True,
       "separate masks; session external, adaptive clock-free")
print("\n[checks] core assertions passed.")


[CHECK] Original Strategy Tester unchanged           PASS - sha-256 vs recorded baseline
[CHECK] Explorer unchanged                           PASS - sha-256 vs recorded baseline
[CHECK] Shared session filter unchanged              PASS - sha-256 vs recorded baseline
[CHECK] Shared feature builder unchanged             PASS - sha-256 vs recorded baseline
[CHECK] Adaptive filter contains no hour restriction PASS 
[CHECK] Rolling features are causal                  PASS - trailing rolling().rank(pct=True); unit-tested
[CHECK] Strategy parameters frozen across scenarios  PASS - single FROZEN_CONFIGS per timeframe
[CHECK] Same backtest engine used across scenarios   PASS - engine hash c335f659b31a
[CHECK] Session and adaptive filters evaluated independently PASS - separate masks; session external, adaptive clock-free

[checks] core assertions passed.


## 14. Filter selection rules, interpretation & final decision

Diagnostics only — never automatic production approval, never ranked by net profit alone.

In [18]:
def _get(df, tf, preset, experiment, segment, col):
    sub = df[(df.timeframe==tf)&(df.filter_preset==preset)&
             (df.experiment_name==experiment)&(df.segment==segment)]
    return float(sub[col].iloc[0]) if len(sub) else np.nan

def evaluate_promotion(results_df, tf, preset="balanced"):
    oos_dd  = _get(results_df, tf, preset, "adaptive_quality", "OOS", "max_drawdown")
    oos_tr  = _get(results_df, tf, preset, "adaptive_quality", "OOS", "trade_count")
    oos_pf  = _get(results_df, tf, preset, "adaptive_quality", "OOS", "profit_factor")
    oos_ppt = _get(results_df, tf, preset, "adaptive_quality", "OOS", "profit_per_trade")
    oos_acc = _get(results_df, tf, preset, "adaptive_quality", "OOS", "candidate_acceptance_rate")
    oos_dom = _get(results_df, tf, preset, "adaptive_quality", "OOS", "dominant_hour_trade_share")
    oos_top3= _get(results_df, tf, preset, "adaptive_quality", "OOS", "top_three_hours_trade_share")
    s_pf    = _get(results_df, tf, preset, "session_only", "OOS", "profit_factor")
    s_ppt   = _get(results_df, tf, preset, "session_only", "OOS", "profit_per_trade")
    s_dd    = _get(results_df, tf, preset, "session_only", "OOS", "max_drawdown")
    min_tr  = MIN_OOS_TRADES.get(tf, 50)

    # DRAWDOWN UNITS: the FROZEN compute_metrics returns max_drawdown ALREADY as a
    # percentage of peak equity (max_dd_pct, x100). It can exceed 100% because the frozen
    # engine permits negative equity (no stop-out). The absolute <=30% gate is therefore
    # effectively UNREACHABLE for this strategy (the session-only baseline DD is itself
    # ~35-47%), so it is demoted to an ADVISORY flag and the RELATIVE adaptive-vs-session
    # comparison becomes the PRIMARY drawdown gate. This is a calibration fix, not OOS
    # tuning: the relative test was always the sounder discriminator (see spec/prior note).
    dd_abs_gate_pass = bool(oos_dd <= MAX_ACCEPTABLE_DD)   # advisory only
    dd_vs_session_ok = bool(oos_dd <= s_dd * 1.10)          # PRIMARY drawdown gate
    # IS->OOS drawdown stability: did the in-sample DD edge over session survive OOS?
    is_dd   = _get(results_df, tf, preset, "adaptive_quality", "IS", "max_drawdown")
    is_s_dd = _get(results_df, tf, preset, "session_only", "IS", "max_drawdown")
    dd_is_advantage  = bool(np.isfinite(is_dd) and np.isfinite(is_s_dd) and is_dd <= is_s_dd * 1.10)
    dd_is_oos_stable = bool(dd_is_advantage and dd_vs_session_ok)
    promising = (dd_vs_session_ok and oos_tr >= min_tr and oos_pf > 1.0
                 and oos_ppt > 0.0 and oos_acc >= 0.25)
    parity = (oos_pf >= 0.95*s_pf and oos_ppt >= 0.90*s_ppt
              and dd_vs_session_ok and oos_acc >= 0.25
              and (not (oos_dom>0.70)) and (not (oos_top3>0.90)))

    if oos_tr < min_tr:
        decision = "INSUFFICIENT_OOS_TRADES"
    elif TIMEZONE_IS_NAIVE and abs(oos_pf - s_pf) < 1e-9 and np.isnan(oos_pf):
        decision = "TIMEZONE_VALIDATION_REQUIRED"
    elif promising and parity:
        decision = "PROMOTE_TO_STRATEGY_TESTER"
    elif promising and not parity:
        decision = "KEEP_AS_RESEARCH_ONLY"
    else:
        decision = "REJECT_FILTER"
    if TIMEZONE_IS_NAIVE and decision == "PROMOTE_TO_STRATEGY_TESTER":
        decision = "TIMEZONE_VALIDATION_REQUIRED"  # naive session labels gate promotion
    return {"timeframe": tf, "preset": preset, "promising": promising, "parity_with_session": parity,
            "oos_pf": oos_pf, "session_pf": s_pf, "oos_ppt": oos_ppt, "session_ppt": s_ppt,
            "oos_dd": oos_dd, "session_dd": s_dd, "oos_trades": oos_tr,
            "dd_abs_gate_pass": dd_abs_gate_pass, "dd_vs_session_ok": dd_vs_session_ok,
            "dd_is_oos_stable": dd_is_oos_stable,
            "oos_acceptance": oos_acc, "oos_dominant_hour": oos_dom, "decision": decision}

# --- IS/CPCV preset selection: choose the TIGHTEST preset that clears the in-sample
# retention floor AND keeps in-sample drawdown within tolerance of session-only.
# OOS is REPORTED on the selected preset; it is never used to select it.
_PRESET_PRIORITY = ["strict", "balanced", "moderate", "loose"]  # tightest -> loosest

def _is_metrics(results_df, tf, preset):
    acc  = _get(results_df, tf, preset, "adaptive_quality", "IS", "candidate_acceptance_rate")
    dd   = _get(results_df, tf, preset, "adaptive_quality", "IS", "max_drawdown")
    s_dd = _get(results_df, tf, preset, "session_only", "IS", "max_drawdown")
    ret_ok = bool(acc >= MIN_IS_RETENTION)
    dd_ok  = bool(np.isfinite(dd) and np.isfinite(s_dd) and dd <= s_dd * DD_SESSION_TOLERANCE)
    return {"timeframe": tf, "filter_preset": preset, "is_retention": acc,
            "is_dd": dd, "session_is_dd": s_dd, "is_retention_ok": ret_ok,
            "is_dd_ok": dd_ok, "is_gate_pass": bool(ret_ok and dd_ok)}

PRESET_IS_SELECTION = pd.DataFrame(
    [_is_metrics(RESULTS_DF, tf, p) for tf in TIMEFRAMES for p in _PRESET_PRIORITY
     if not np.isnan(_get(RESULTS_DF, tf, p, "adaptive_quality", "IS", "candidate_acceptance_rate"))]
)
print("=== IS/CPCV PRESET SELECTION (in-sample tuning) ===")
print("    gate: retention >= {:.0%} AND in-sample DD <= {:.2f}x session-only DD".format(MIN_IS_RETENTION, DD_SESSION_TOLERANCE))
if len(PRESET_IS_SELECTION):
    display(PRESET_IS_SELECTION)

def _select_preset(tf):
    for p in _PRESET_PRIORITY:  # tightest first: keep max quality that still clears retention+DD
        row = PRESET_IS_SELECTION[(PRESET_IS_SELECTION.timeframe==tf) & (PRESET_IS_SELECTION.filter_preset==p)]
        if len(row) and bool(row.iloc[0]["is_gate_pass"]):
            return p, True
    return "balanced", False

SELECTED_PRESETS = {}
for tf in TIMEFRAMES:
    _p, _cleared = _select_preset(tf)
    SELECTED_PRESETS[tf] = _p
    print("[select] {}: preset='{}'{}".format(tf, _p, "" if _cleared else "  (NO preset cleared the IS retention+DD gate; falling back to 'balanced')"))

DECISIONS = pd.DataFrame([evaluate_promotion(RESULTS_DF, tf, SELECTED_PRESETS[tf]) for tf in TIMEFRAMES])
display(DECISIONS)

print("\n=== REQUIRED INTERPRETATION (per timeframe) ===")
for tf in TIMEFRAMES:
    prof = _raw.get(tf)
    total = int(prof.sum()) if prof is not None else 0
    pre15 = int(prof.loc[prof.index<15].sum()) if prof is not None else 0
    print(f"\n[{tf}]")
    print(f"  1. Raw candidates throughout the day? total={total}, before 15:00={pre15}"
          f" ({(pre15/max(total,1)):.0%}).")
    print("  2/3. 13-15h concentration present BEFORE session filtering: read section 11 table.")
    print("       Attribution across raw-signal / regime / session / adaptive / occupancy:")
    print("       compare baseline_all_day vs session_only vs adaptive_quality acceptance + hourly.")
    print("  4. Component attribution: inspect candidate_funnel (after_volume/atr/adx) in FUNNELS_DF.")
    d = DECISIONS[DECISIONS.timeframe==tf].iloc[0]
    print(f"  5. Adaptive vs session-only OOS PF: {d.oos_pf} vs {d.session_pf} (parity={d.parity_with_session}).")
    print(f"     Drawdown is % of peak equity (frozen engine; may exceed 100% as it allows")
    print(f"     negative equity): adaptive {d.oos_dd:.1f}% vs session {d.session_dd:.1f}%"
          f" | abs<=30% advisory={d.dd_abs_gate_pass} | PRIMARY not-worse-than-session={d.dd_vs_session_ok}.")
    print(f"     IS->OOS DD stability={d.dd_is_oos_stable} (True only if adaptive's in-sample DD edge over session persisted OOS).")
    print(f"  6. Trades outside dominant hours: dominant_hour_share={d.oos_dominant_hour}.")
    print("  7. session_plus_adaptive vs adaptive_quality: compare rows to detect over-filtering.")
    print("  8. Stability across loose/moderate/balanced/strict: compare preset rows (reject isolated optima).")
    print("     Selected preset for {} (IS-tuned): {}.".format(tf, DECISIONS[DECISIONS.timeframe==tf].iloc[0]['preset']))
    print(f"  9. Predictive vs mere frequency reduction: acceptance={d.oos_acceptance}, ppt={d.oos_ppt}.")
    print(f"  10. DECISION: {d.decision}")

print("\nNOTE: A decision is only meaningful once run on the REAL data with numba/xgboost.")
print("Do NOT modify Strategy Tester or Explorer automatically, even if positive.")
print("Preset was selected on IN-SAMPLE retention+DD only (see IS selection table above); the OOS decision is out-of-sample.")


=== IS/CPCV PRESET SELECTION (in-sample tuning) ===
    gate: retention >= 25% AND in-sample DD <= 1.10x session-only DD


,timeframe,filter_preset,is_retention,is_dd,session_is_dd,is_retention_ok,is_dd_ok,is_gate_pass
0,M15,strict,0.054995,42.768508,42.272673,False,True,False
1,M15,balanced,0.139244,38.957472,42.272673,False,True,False
2,M15,moderate,0.355896,33.763316,42.272673,True,True,True
3,M15,loose,0.459496,34.889198,42.272673,True,True,True
4,M5,strict,0.040495,36.687064,47.325618,False,True,False
5,M5,balanced,0.102031,34.614763,47.325618,False,True,False
6,M5,moderate,0.344680,36.979023,47.325618,True,True,True
7,M5,loose,0.437467,36.774932,47.325618,True,True,True


[select] M15: preset='moderate'
[select] M5: preset='moderate'


,timeframe,preset,promising,parity_with_session,oos_pf,session_pf,oos_ppt,session_ppt,oos_dd,session_dd,oos_trades,dd_abs_gate_pass,dd_vs_session_ok,dd_is_oos_stable,oos_acceptance,oos_dominant_hour,decision
0,M15,moderate,False,False,0.000000,0.000000,-106.034154,-105.974640,42.413662,42.389856,6.0,False,True,True,0.355433,0.333333,INSUFFICIENT_OOS_TRADES
1,M5,moderate,False,False,1.637418,1.529865,13.595751,10.199868,43.296682,35.514647,1684.0,False,False,False,0.356788,0.077791,REJECT_FILTER



=== REQUIRED INTERPRETATION (per timeframe) ===

[M15]
  1. Raw candidates throughout the day? total=14202, before 15:00=10340 (73%).
  2/3. 13-15h concentration present BEFORE session filtering: read section 11 table.
       Attribution across raw-signal / regime / session / adaptive / occupancy:
       compare baseline_all_day vs session_only vs adaptive_quality acceptance + hourly.
  4. Component attribution: inspect candidate_funnel (after_volume/atr/adx) in FUNNELS_DF.
  5. Adaptive vs session-only OOS PF: 0.0 vs 0.0 (parity=False).
     Drawdown is % of peak equity (frozen engine; may exceed 100% as it allows
     negative equity): adaptive 42.4% vs session 42.4% | abs<=30% advisory=False | PRIMARY not-worse-than-session=True.
     IS->OOS DD stability=True (True only if adaptive's in-sample DD edge over session persisted OOS).
  6. Trades outside dominant hours: dominant_hour_share=0.3333333333333333.
  7. session_plus_adaptive vs adaptive_quality: compare rows to detect over-f

## 15. Confirm source notebooks & shared code were not modified (SHA-256)

In [19]:
def _sha(path):
    p = Path(path)
    return hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None

_bundle = Path(_repo_root) / "pipeline_verification_bundle"
_targets = {
    "Strategy_Tester.ipynb": _bundle/"Strategy_Tester.ipynb",
    "GoldRegimeX_Explorer.ipynb": _bundle/"GoldRegimeX_Explorer.ipynb",
    "shared/session_filter.py": _bundle/"shared"/"session_filter.py",
    "shared/features.py": _bundle/"shared"/"features.py",
}
print("[integrity] SHA-256 of protected source files vs recorded baseline:")
for name, p in _targets.items():
    _cur = _sha(p); _base = BASELINE_HASHES.get(name)
    _status = "MATCH" if (_cur and _base and _cur == _base) else ("DIFFERS" if _cur else "MISSING")
    print(f"   {name:32s} {_status:8s} {_cur}")
# Unchanged-flags are computed in the section-13 checks cell against BASELINE_HASHES;
# by construction this lab only writes to experiments/*.py and reports/*.csv.
print("\n[integrity] This lab only writes to experiments/*.py and reports/adaptive_filter_*.csv.")


[integrity] SHA-256 of protected source files vs recorded baseline:
   Strategy_Tester.ipynb            MATCH    ed78f5b7f373841e056a422568443af658c18c2ae06c87cc1494275e5b896ffc
   GoldRegimeX_Explorer.ipynb       MATCH    d40beb6b57a817c252ca2f341216a0f19f096255c665071423f6864cd5d1755d
   shared/session_filter.py         MATCH    2bb549be91b48a514921ac568ac2dd83f97a2ad890cabf6d2c68f5ce847a9552
   shared/features.py               MATCH    63da75dc10a24253acba2684c49cf86d35fbdbcf864033e16cc8abbfe7643861

[integrity] This lab only writes to experiments/*.py and reports/adaptive_filter_*.csv.


## 16. FUTURE TRANSFER PLAN (provided, NOT executed)

**Do not modify Strategy Tester or Explorer during this experiment**, even if the
decision is positive. The following is a plan only.

### Strategy Tester integration point
```python
routed_signals = generate_routed_signals(df, entry_params, strategy_name)
quality_mask   = strategy_quality_mask(df, strategy_name, frozen_filter_config)
final_signals  = routed_signals.where(quality_mask, 0)
```
Gate with independent toggles so the two filters remain separable:
```python
ENABLE_SESSION_FILTER  = False
ENABLE_ADAPTIVE_FILTER = True
```
Steps:
1. Add `experiments/adaptive_filter_lab.py` as an import (already isolated).
2. Add the two toggles + `frozen_filter_config` next to the config constants.
3. Insert the `quality_mask` application strictly **after** `generate_routed_signals`
   and **before** `run_backtest`; leave the engine, grids, stops, targets, exits,
   regime routing and session definitions untouched.
4. Re-run the full grid; require the adaptive OOS result to satisfy the section-18
   parity rules before making it a default.

### Explorer integration point
```
Rule-based signal -> Regime routing -> Adaptive market-quality mask ->
XGBoost directional probability / margin -> Backtest engine
```
- Use the adaptive mask **only as a post-model eligibility gate** first, so its
  effect stays attributable. Do **not** add adaptive features to the XGBoost training
  matrix initially.
- Explore `ML_MARGIN_GRID` with `directional_probability_margin` / `ml_margin_mask`
  on IS/CPCV only; a directional *margin* may discriminate better than the current
  low `probability >= 0.15` threshold, which appears inert.

### Live bundle (later; do NOT edit during this phase)
```python
"adaptive_filter": {"enabled": True, "config": frozen_filter_config, "version": "adaptive_quality_v1"}
```

### Timezone prerequisite
Because session labels are applied to naive (likely broker) timestamps, resolve the
broker UTC offset and re-validate session parity **before** any promotion that relies
on session comparison (decision `TIMEZONE_VALIDATION_REQUIRED`).
